In [2]:
import random
import numpy as np
import gym
from collections import defaultdict

In [3]:
class CardGameEnv(gym.Env):
    def __init__(self):
        self.num_decks = 3
        self.num_cards_per_deck = 52
        self.num_cards_per_player = 21
        self.deck = self.reset_deck()
        self.joker = self.deck.pop() % 52
        self.hand = self.reset_hand()
        
        self.discard_pile = self.reset_discard_pile()
    
    def reset_discard_pile(self):
        """
        Set the top card of the discard pile.
        """
        discard_pile = [0 for _ in range(self.num_cards_per_deck)]
        
        current_card = self.deck.pop() % 52
        discard_pile[current_card] = 1
        
        return discard_pile

    def reset_deck(self):
        """
        Initialize the deck with 3 decks of 52 cards.
        """
        deck = list(range(self.num_cards_per_deck * self.num_decks))
        random.shuffle(deck)
        return deck

    def reset_hand(self):
        """
        Deal cards to the player
        """
        hand = [0 for _ in range(self.num_cards_per_deck)]
        
        for _ in range(self.num_cards_per_player):
            current_card = self.deck.pop() % 52
            hands[current_card] += 1

        return hand

    def encode_state(self):
        """
        Encode the player's hand and discard pile as one-hot vectors.
        """
        pile_encoding = np.zeros(self.num_cards_per_deck)
        for card_seen in self.discard_pile:
            pile_encoding[card_seen] += 1

        return np.concatenate([hand_encoding, pile_encoding, np.array([self.joker])])  # Shape (52 cards * 2 + 1)

    def step(self, action):
        """
        Action: 0 = Draw a new card, 1 = Pick up from discard pile
        """
        reward = 0
        if action == 1:  # Pick up discard
            picked_card = self.discard_pile.pop()  # Take the top discard card
            
            if self.is_bad_card(card):
                reward = -10
            else:
                reward = 0
            
        else:  # Draw from deck
            if len(self.deck == 0):
                discarded_cards = set()
                for card in self.discard_pile:
                    if (card + 52) in discarded_cards:
                        discarded_cards.add(card + 52 + 52)
                    elif card in discarded_cards:
                        discarded_cards.add(card + 52)
                    else:
                        discarded_cards.add(card)
                self.deck = random.shuffle(discarded_cards)
                self.discard_pile = []
                
            new_card = self.deck.pop() % 52
            
            if self.is_bad_card(card):
                reward = 0
            else:
                reward = 5
            
        if new_card == self.joker:
            reward = 20
    
        # Game progression rewards
        if self.is_game_won():
            reward += 100

        return self.encode_state(), reward, False, {}

    def is_bad_card(self, card):
        """
        Determines if a card is a bad card to pick up.
        A bad card is one that does not help form a valid sequence or set.
        """
        if card == self.joker:  # Joker is always useful
            return False


        # Check if adding the card forms a valid set or sequence
        # A set of 3 or more cards of the same rank
        if self.hand.count(card) >= 2:  # Already two cards of the same rank
            return False

        # Check how useful the card is

        # if a card does not have any near neighbors or a card of the same rank
        if (card - 1 not in self.hand and card + 1 not in self.hand and card % 13 not in self.hand):
            return True

        return False

    def is_game_won(self):
        return len(self.hand) == 0  # Win condition: no cards left

SyntaxError: invalid syntax (1003828286.py, line 15)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, action_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class DQNAgent:
    def __init__(self, state_size, action_size, gamma=0.99, epsilon=0.1, learning_rate=0.001):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma  # Discount factor for future rewards
        self.epsilon = epsilon  # Exploration rate
        self.learning_rate = learning_rate
        
        # Initialize the Q-network
        self.q_network = DQN(state_size, action_size)
        self.target_network = DQN(state_size, action_size)  # Target network for stability
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=self.learning_rate)
        self.loss_fn = nn.MSELoss()

        # Initialize the target network with the same weights as the Q-network
        self.target_network.load_state_dict(self.q_network.state_dict())
        
    def select_action(self, state):
        """Select action using epsilon-greedy policy."""
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.action_size)  # Exploration
        else:
            state = torch.FloatTensor(state).unsqueeze(0)
            q_values = self.q_network(state)
            return torch.argmax(q_values).item()  # Exploitation
    
    def update(self, state, action, reward, next_state, done):
        """Update Q-network using Q-learning."""
        state = torch.FloatTensor(state).unsqueeze(0)
        next_state = torch.FloatTensor(next_state).unsqueeze(0)
        
        # Get the predicted Q-value for the current state and action
        q_value = self.q_network(state)[0][action]
        
        # Get the maximum Q-value from the next state
        with torch.no_grad():
            max_next_q_value = torch.max(self.target_network(next_state))
        
        # Calculate the target value
        target = reward + (1 - done) * self.gamma * max_next_q_value
        
        # Compute the loss and perform the optimization step
        loss = self.loss_fn(q_value, target)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
    def update_target_network(self):
        """Update target network with the Q-network's weights."""
        self.target_network.load_state_dict(self.q_network.state_dict())


In [5]:
class MCTS:
    def __init__(self, agent, c=1.0, num_simulations=100):
        self.agent = agent
        self.c = c  # Exploration factor
        self.num_simulations = num_simulations

    def search(self, state):
        """
        Perform MCTS to find the best action using the DQN model as the evaluation function.
        """
        root = Node(state)
        
        for _ in range(self.num_simulations):
            leaf = self._select(root)  # Select a leaf node to expand
            reward = self._simulate(leaf)  # Simulate the game from this node to the end
            self._backpropagate(leaf, reward)  # Backpropagate the reward through the tree

        # Choose the action with the highest visit count (best action)
        return self._best_action(root)

    def _select(self, node):
        """Select a node using UCB1 (Upper Confidence Bound)."""
        while node.is_fully_expanded():
            node = node.best_child(self.c)
        return node.expand()  # Expand the selected node

    def _simulate(self, node):
        """Simulate the game to the end from the current node's state."""
        state = node.state
        done = False
        total_reward = 0

        while not done:
            action = self.agent.select_action(state)
            next_state, reward, done, _ = state.step(action)  # Get next state after action
            total_reward += reward
            state = next_state

        return total_reward

    def _backpropagate(self, node, reward):
        """Backpropagate the simulation result up the tree."""
        while node is not None:
            node.update(reward)
            node = node.parent

    def _best_action(self, root):
        """Choose the best action from the root node."""
        return root.best_action()


In [6]:
class Node:
    def __init__(self, state):
        self.state = state  # The current state
        self.parent = None  # Parent node
        self.children = []  # Child nodes
        self.visits = 0  # Number of times this node has been visited
        self.total_reward = 0  # Sum of rewards from simulations
        self.untried_actions = [0, 1]  # Possible actions: draw or pick up
        self.action = None  # Action leading to this state

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def best_child(self, c):
        """Select the child with the highest UCB1 value."""
        return max(self.children, key=lambda child: child.total_reward / (child.visits + 1e-6) + c * np.sqrt(np.log(self.visits + 1) / (child.visits + 1e-6)))

    def expand(self):
        """Expand this node by trying a new action."""
        action = self.untried_actions.pop()
        next_state, _, _, _ = self.state.step(action)
        child_node = Node(next_state)
        child_node.parent = self
        child_node.action = action
        self.children.append(child_node)
        return child_node

    def update(self, reward):
        """Update this node with the result of a simulation."""
        self.visits += 1
        self.total_reward += reward
